# nnU-Net Colab Runner
Requires `config.py`, `prepare_dataset.py`, `preprocess.py`, `train.py`, `evaluate.py` in GitHub repo or uploaded directly into Colab's `/content/` directory, in created folder `src`.  

## 1. Mount Drive & Pull Latest Code

In [ ]:
### GITHUB VERSION

import os
import sys
import torch
from google.colab import drive, userdata

# Mount Google Drive
drive.mount('/content/drive')

# Configure GitHub authentication and repo paths
GITHUB_ORG = "rayray2006"
GITHUB_REPO = "BDDProject7"
REPO_DIR = f"/content/{GITHUB_REPO}"

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    repo_url = f"https://{token}@github.com/{GITHUB_ORG}/{GITHUB_REPO}.git"
else:
    repo_url = f"https://github.com/{GITHUB_ORG}/{GITHUB_REPO}.git"

# Clone or update repository
if not os.path.exists(REPO_DIR):
    !git clone "{repo_url}" "{REPO_DIR}"
else:
    %cd "{REPO_DIR}"
    # Reset local edits to prevent git pull merge conflicts from breaking the cell
    !git fetch origin
    !git reset --hard origin/main
    !git pull

# Set working directory and register module path
%cd "{REPO_DIR}"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install dependencies and verify hardware
!pip install --quiet -r requirements.txt
!pip install --quiet nnunetv2 python-dotenv typer rich nibabel ipython --upgrade

assert torch.cuda.is_available(), "CRITICAL: CUDA is not active! Go to Runtime > Change runtime type > Select T4 GPU."
print(f"CUDA Active: {torch.cuda.get_device_name(0)}")

# Enable hot-reloading
%load_ext autoreload
%autoreload 2

In [ ]:
### NON-GITHUB VERSION
# CHECK: .py files on local computer and uploaded under 'src' folder in Colab file directory.
import os
import sys
import torch
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Ensure root directory and register Python search path
%cd /content
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

# Install dependencies and verify hardware
!pip install --quiet nnunetv2 python-dotenv typer rich nibabel ipython --upgrade

assert torch.cuda.is_available(), (
    "CRITICAL: CUDA is not active!\n"
    "Go to Runtime > Change runtime type > Select T4 GPU (or other GPU accelerator)."
)
print(f"CUDA Active: {torch.cuda.get_device_name(0)}")

# Enable hot-reloading for edited .py files
%load_ext autoreload
%autoreload 2

## 2. Prepare Datasets
Separate 45 OARs and 2 GTVs

In [ ]:
# To test n number of cases, add to the end of GENERATE DATASET commands: --max-cases n
import os

# Define Google Drive paths. base_dir is the output folder; data_dir is the path to the full dataset.
base_dir = "/content/drive/MyDrive/.../SegRap2023_Xcases"
data_dir = "/content/drive/MyDrive/.../SegRap2023_Training_Set_120cases"

os.environ["nnUNet_raw"] = f"{base_dir}/nnunet/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{base_dir}/nnunet/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{base_dir}/nnunet/nnUNet_results"

# Generate dataset 501: 45 OARs (2-channel ncCT + ceCT)
!python -m src.prepare_dataset --raw-source-dir "{data_dir}" --task oar
# Generate dataset 502: 2 GTVs (2-channel ncCT + ceCT)
!python -m src.prepare_dataset --raw-source-dir "{data_dir}" --task gtv

## 3. Plan and Preprocess

In [ ]:
# Verifies integrity, calculates foreground percentiles, resamples anisotropic spacing
# Update --num-processes depending on available compute
!python -m src.preprocess --dataset-id 501 --configuration 3d_fullres --num-processes 4
!python -m src.preprocess --dataset-id 502 --configuration 3d_fullres --num-processes 4

## 4. Training
Standard 3d_fullres with compound dice and cross-entropy loss

In [ ]:
# Train folds (0-4) sequentially or across multiple team member accounts
# (This may have to be adapted to parallel processing for the cluster)
FOLD = 0

# Train Dataset 501 (OARs)
# Add -c to resume from checkpoint_latest.pth if disconnected
!python -m src.train --dataset-id 501 --configuration 3d_fullres --fold {FOLD}

In [ ]:
# Separate cell for training across multiple team member accounts (and control for Colab Pro's 24-hour runtime limit)
FOLD = 0

# Train Dataset 502 (GTVs)
# Add -c to resume from checkpoint_latest.pth if disconnected
!python -m src.train --dataset-id 502 --configuration 3d_fullres --fold {FOLD}

## 5. Find Best Configuration and Determine Postprocessing
Do NOT run this cell until ALL 5 FOLDS (0, 1, 2, 3, 4) have completed training for the target dataset, and their validation files exist in Drive.

In [ ]:
# Ensembles folds and searches for optimal connected component filtering
!python -m src.evaluate optimize-and-postprocess --dataset-id 501 --configuration 3d_fullres
!python -m src.evaluate optimize-and-postprocess --dataset-id 502 --configuration 3d_fullres

## 6. Multi-Fold Ensembled Inference

In [ ]:
TEST_IMAGES = f"{base_dir}/test_images"
PRED_OAR = f"{base_dir}/predictions/oar"
PRED_GTV = f"{base_dir}/predictions/gtv"

# Run 5-fold ensembled prediction (requires Folds 0-4 completed)
!python -m src.evaluate predict --dataset-id 501 --input-folder {TEST_IMAGES} --output-folder {PRED_OAR} --folds 0,1,2,3,4
!python -m src.evaluate predict --dataset-id 502 --input-folder {TEST_IMAGES} --output-folder {PRED_GTV} --folds 0,1,2,3,4

In [ ]:
### OPTIONAL: Test prediction on a single fold using the best saved checkpoint
INPUT_SAMPLE = f"{base_dir}/nnunet/nnUNet_raw/Dataset501_SegRAP2023_OAR/imagesTr"
OUTPUT_SAMPLE = f"{base_dir}/predictions_test"

!nnUNetv2_predict -i "{INPUT_SAMPLE}" -o "{OUTPUT_SAMPLE}" -d 501 -c 3d_fullres -f 0 -chk checkpoint_best.pth -device cuda --not_on_device -npp 2 -nps 2